In [1]:
import numpy as np
import pandas as pd

In [2]:
## -----------------------------
# 1. DATASET
# -----------------------------
df=pd.read_csv('titanic.csv')


In [3]:
# ============================================================
# 2. SELECT REQUIRED COLUMNS
# ============================================================

data = df[[
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "Survived"
]].copy()


# ============================================================
# 3. HANDLE MISSING VALUES
# ============================================================

data["Age"] = data["Age"].fillna(data["Age"].median())


# ============================================================
# 4. CONVERT SEX TO NUMERICAL VALUE
# ============================================================

data["Sex"] = data["Sex"].map({
    "male": 0,
    "female": 1
})


# ============================================================
# 5. SEPARATE FEATURES AND TARGET
# ============================================================

X = data[[
    "Pclass",
    "Sex",
    "Age",
    "Fare"
]]

y = data["Survived"]

In [4]:
# ============================================================
# 6. ENTROPY
# ============================================================

def entropy(y):

    values, counts = np.unique(
        y,
        return_counts=True
    )

    probabilities = counts / len(y)

    return -np.sum(
        probabilities * np.log2(probabilities)
    )


# ============================================================
# 7. INFORMATION GAIN
# ============================================================

def information_gain(parent_y, left_y, right_y):

    parent_entropy = entropy(parent_y)

    left_weight = len(left_y) / len(parent_y)
    right_weight = len(right_y) / len(parent_y)

    child_entropy = (
        left_weight * entropy(left_y)
        +
        right_weight * entropy(right_y)
    )

    gain = parent_entropy - child_entropy

    return gain


# ============================================================
# 8. FIND BEST SPLIT
# ============================================================

def find_best_split(X, y):

    best_gain = -1
    best_feature = None
    best_threshold = None

    for feature in X.columns:

        values = np.sort(
            X[feature].unique()
        )

        # ----------------------------------------------------
        # Create candidate thresholds
        # ----------------------------------------------------

        thresholds = []

        for i in range(len(values) - 1):

            threshold = (
                values[i] + values[i + 1]
            ) / 2

            thresholds.append(threshold)


        # ----------------------------------------------------
        # Try every threshold
        # ----------------------------------------------------

        for threshold in thresholds:

            left_mask = X[feature] <= threshold
            right_mask = X[feature] > threshold

            left_y = y[left_mask]
            right_y = y[right_mask]

            # Ignore empty branches
            if len(left_y) == 0 or len(right_y) == 0:
                continue

            gain = information_gain(
                y,
                left_y,
                right_y
            )

            if gain > best_gain:

                best_gain = gain
                best_feature = feature
                best_threshold = threshold


    return (
        best_feature,
        best_threshold,
        best_gain
    )


# ============================================================
# 9. MAJORITY CLASS
# ============================================================

def majority_class(y):

    values, counts = np.unique(
        y,
        return_counts=True
    )

    return int(
        values[np.argmax(counts)]
    )


# ============================================================
# 10. BUILD ID3 TREE
# ============================================================

def build_tree(
    X,
    y,
    depth=0,
    max_depth=4,
    min_samples=5
):

    # --------------------------------------------------------
    # CASE 1:
    # All samples belong to one class
    # --------------------------------------------------------

    if len(np.unique(y)) == 1:

        return int(y.iloc[0])


    # --------------------------------------------------------
    # CASE 2:
    # No samples
    # --------------------------------------------------------

    if len(y) == 0:

        return 0


    # --------------------------------------------------------
    # CASE 3:
    # Maximum depth reached
    # --------------------------------------------------------

    if depth >= max_depth:

        return majority_class(y)


    # --------------------------------------------------------
    # CASE 4:
    # Too few samples
    # --------------------------------------------------------

    if len(y) < min_samples:

        return majority_class(y)


    # --------------------------------------------------------
    # FIND BEST SPLIT
    # --------------------------------------------------------

    feature, threshold, gain = find_best_split(
        X,
        y
    )


    # --------------------------------------------------------
    # CASE 5:
    # No useful split
    # --------------------------------------------------------

    if feature is None or gain <= 0:

        return majority_class(y)


    # --------------------------------------------------------
    # SPLIT DATA
    # --------------------------------------------------------

    left_mask = X[feature] <= threshold
    right_mask = X[feature] > threshold


    X_left = X[left_mask]
    y_left = y[left_mask]

    X_right = X[right_mask]
    y_right = y[right_mask]


    # --------------------------------------------------------
    # RECURSIVELY BUILD LEFT TREE
    # --------------------------------------------------------

    left_tree = build_tree(
        X_left,
        y_left,
        depth + 1,
        max_depth,
        min_samples
    )


    # --------------------------------------------------------
    # RECURSIVELY BUILD RIGHT TREE
    # --------------------------------------------------------

    right_tree = build_tree(
        X_right,
        y_right,
        depth + 1,
        max_depth,
        min_samples
    )


    # --------------------------------------------------------
    # RETURN TREE NODE
    # --------------------------------------------------------

    return {
        "feature": feature,
        "threshold": threshold,
        "gain": gain,
        "left": left_tree,
        "right": right_tree
    }


# ============================================================
# 11. BOOTSTRAP SAMPLE
# ============================================================

def bootstrap_sample(X, y):

    n = len(X)

    random_indices = np.random.choice(
        n,
        size=n,
        replace=True
    )

    X_sample = X.iloc[
        random_indices
    ].reset_index(drop=True)

    y_sample = y.iloc[
        random_indices
    ].reset_index(drop=True)

    return X_sample, y_sample


# ============================================================
# 12. TRAIN ONE RANDOM TREE
# ============================================================

def train_tree(X, y):

    X_sample, y_sample = bootstrap_sample(
        X,
        y
    )

    tree = build_tree(
        X_sample,
        y_sample,
        max_depth=4,
        min_samples=5
    )

    return tree


# ============================================================
# 13. PREDICT ONE PASSENGER
# ============================================================

def predict_one(tree, passenger):

    # --------------------------------------------------------
    # Leaf node
    # --------------------------------------------------------

    if not isinstance(tree, dict):

        return tree


    feature = tree["feature"]
    threshold = tree["threshold"]


    # --------------------------------------------------------
    # Follow left branch
    # --------------------------------------------------------

    if passenger[feature] <= threshold:

        return predict_one(
            tree["left"],
            passenger
        )


    # --------------------------------------------------------
    # Follow right branch
    # --------------------------------------------------------

    else:

        return predict_one(
            tree["right"],
            passenger
        )


# ============================================================
# 14. RANDOM FOREST PREDICTION
# ============================================================

def random_forest(
    passenger,
    trees
):

    predictions = []

    for tree in trees:

        prediction = predict_one(
            tree,
            passenger
        )

        predictions.append(prediction)


    # --------------------------------------------------------
    # MAJORITY VOTING
    # --------------------------------------------------------

    predictions = np.array(
        predictions
    )

    final_prediction = np.bincount(
        predictions
    ).argmax()


    return predictions, final_prediction


# ============================================================
# 15. PRINT TREE
# ============================================================

def print_tree(
    tree,
    depth=0
):

    spacing = "    " * depth

    # --------------------------------------------------------
    # Leaf
    # --------------------------------------------------------

    if not isinstance(tree, dict):

        print(
            spacing +
            "Prediction:",
            tree
        )

        return


    feature = tree["feature"]
    threshold = tree["threshold"]
    gain = tree["gain"]


    print(
        spacing +
        f"{feature} <= {threshold:.2f}"
        +
        f"  [Gain={gain:.4f}]"
    )


    print(
        spacing +
        "├── YES:"
    )

    print_tree(
        tree["left"],
        depth + 1
    )


    print(
        spacing +
        "└── NO:"
    )

    print_tree(
        tree["right"],
        depth + 1
    )


# ============================================================
# 16. BUILD RANDOM FOREST
# ============================================================

np.random.seed(42)

number_of_trees = 4

trees = []


for i in range(number_of_trees):

    tree = train_tree(
        X,
        y
    )

    trees.append(tree)


# ============================================================
# 17. DISPLAY FOUR TREES
# ============================================================

for i, tree in enumerate(trees):

    print("\n")
    print("=" * 60)
    print(f"TREE {i + 1}")
    print("=" * 60)

    print_tree(tree)


# ============================================================
# 18. TEST FIRST 10 PASSENGERS
# ============================================================

print("\n")
print("=" * 70)
print("FIRST 10 PASSENGERS")
print("=" * 70)


correct = 0


for i in range(10):

    passenger = X.iloc[i]

    actual = int(
        y.iloc[i]
    )


    predictions, final_prediction = random_forest(
        passenger,
        trees
    )


    if final_prediction == actual:

        correct += 1


    print(
        f"\nPassenger {i + 1}"
    )

    print(
        "Tree predictions:",
        predictions
    )

    print(
        "Final prediction:",
        final_prediction
    )

    print(
        "Actual:",
        actual
    )


# ============================================================
# 19. ACCURACY ON FIRST 10
# ============================================================

accuracy = (
    correct / 10
) * 100


print("\n")
print("=" * 70)

print(
    f"Accuracy on first 10 passengers: "
    f"{accuracy:.2f}%"
)

print("=" * 70)


# ============================================================
# 20. CREATE YOUR OWN PASSENGER
# ============================================================

my_passenger = pd.Series({

    "Pclass": 2,

    "Sex": 0,       # Male

    "Age": 22,

    "Fare": 30
})


# ============================================================
# 21. PREDICT YOUR PASSENGER
# ============================================================

predictions, final_prediction = random_forest(
    my_passenger,
    trees
)


print("\n")
print("=" * 70)
print("MY PASSENGER")
print("=" * 70)


print(
    "\nPassenger details:"
)

print(
    my_passenger
)


print(
    "\nPredictions from 4 trees:",
    predictions
)


print(
    "Final prediction:",
    final_prediction
)


if final_prediction == 1:

    print(
        "Result: Survived"
    )

else:

    print(
        "Result: Did Not Survive"
    )



TREE 1
Sex <= 0.50  [Gain=0.2208]
├── YES:
    Age <= 9.50  [Gain=0.0230]
    ├── YES:
        Fare <= 19.91  [Gain=0.4876]
        ├── YES:
            Prediction: 1
        └── NO:
            Pclass <= 2.50  [Gain=0.8366]
            ├── YES:
                Prediction: 1
            └── NO:
                Prediction: 0
    └── NO:
        Fare <= 26.27  [Gain=0.0260]
        ├── YES:
            Fare <= 23.35  [Gain=0.0154]
            ├── YES:
                Prediction: 0
            └── NO:
                Prediction: 0
        └── NO:
            Age <= 60.50  [Gain=0.0550]
            ├── YES:
                Prediction: 0
            └── NO:
                Prediction: 0
└── NO:
    Pclass <= 2.50  [Gain=0.2152]
    ├── YES:
        Fare <= 149.04  [Gain=0.0182]
        ├── YES:
            Fare <= 29.36  [Gain=0.0332]
            ├── YES:
                Prediction: 1
            └── NO:
                Prediction: 1
        └── NO:
            Fare <= 152.51  [Gain=0.469